In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score as ari_score

# -----------------------------
# Settings
# -----------------------------
sc.settings.verbosity = 0
sns.set(style="whitegrid")

ground_truth_path = "Data/adata_raw_qc.h5ad"   # ground truth file
imputed_root = "imputed_h5ad"   

imputed_dirs = {
    "SoftImpute": "imputed_softimpute",
    "MAGIC": "imputed_h5ad",
    "KNN": "imputed_knn",
    "Mean": "imputed_mean",
    "GAN": "imputed_gan",
    "Iterative": "imputed_iterative"
}               # parent folder with subfolders per method

# Example folder structure:
# imputed_h5ad/
#   ├── gan/
#   ├── knn/
#   ├── softimpute/
#   └── iterative/

# -----------------------------
# Preprocessing function
# -----------------------------
def preprocess_adata(adata, n_hvgs=1000):
    adata = adata.copy()
    adata.obs_names_make_unique()
    adata.var_names_make_unique()

    X = adata.X.A if hasattr(adata.X, "A") else adata.X
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    adata.X = X

    if np.max(adata.X) > 30:
        sc.pp.log1p(adata)

    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvgs)
    adata = adata[:, adata.var["highly_variable"]].copy()

    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)

    return adata

# -----------------------------
# Load ground truth
# -----------------------------
print(" Preprocessing ground truth...")
adata_gt = sc.read_h5ad(ground_truth_path)
adata_gt = preprocess_adata(adata_gt)

if "leiden" not in adata_gt.obs:
    raise ValueError("Ground truth AnnData does not have 'leiden' clustering.")

# -----------------------------
# Evaluate imputation runs
# -----------------------------
# -----------------------------
# Evaluate imputation runs across all methods
# -----------------------------
results = []

for method, method_dir in imputed_dirs.items():
    if not os.path.isdir(method_dir):
        print(f"Skipping {method}, folder not found: {method_dir}")
        continue

    print(f"\n Processing method: {method}")
    for fname in os.listdir(method_dir):
        if not fname.endswith(".h5ad"):
            continue

        fpath = os.path.join(method_dir, fname)
        try:
            adata_imp = sc.read_h5ad(fpath)
            adata_imp = preprocess_adata(adata_imp)

            # Align cells
            common_cells = adata_gt.obs_names.intersection(adata_imp.obs_names)
            adata_gt_sub = adata_gt[common_cells]
            adata_imp_sub = adata_imp[common_cells]

            # Compute ARI
            ari = ari_score(adata_gt_sub.obs["leiden"], adata_imp_sub.obs["leiden"])

            # Parse metadata from filename
            mf, run = None, None
            for part in fname.split("_"):
                if part.startswith("mf"):
                    mf = float(part.replace("mf", "").replace(".h5ad", "")) / 100
                if part.startswith("run"):
                    run = int(part.replace("run", "").replace(".h5ad", ""))

            results.append({
                "file": fname,
                "method": method,
                "ARI": ari,
                "missing_fraction": mf,
                "run": run
            })

            print(f"{method} | {fname}: ARI={ari:.3f}")

        except Exception as e:
            print(f"Error with {fname}: {e}")

# -----------------------------
# Results → DataFrame
# -----------------------------
results_df = pd.DataFrame(results)
if results_df.empty:
    raise ValueError("No results computed. Check your imputed files and pipeline.")

results_df = results_df.dropna(subset=["ARI", "missing_fraction"])
results_df["missing_fraction"] = results_df["missing_fraction"].astype(float)

# -----------------------------
# Summary
# -----------------------------
summary = results_df.groupby(["method", "missing_fraction"]).agg(
    ARI_mean=("ARI", "mean"),
    ARI_std=("ARI", "std"),
    runs=("ARI", "count")
).reset_index()

print("\nSummary (mean ± std):")
print(summary)

# -----------------------------
# Plot: ARI vs Missing Fraction across Methods
# -----------------------------
plt.figure(figsize=(10, 6))
sns.lineplot(
    data=results_df,
    x="missing_fraction",
    y="ARI",
    hue="method",
    marker="o",
    err_style="band",  # shaded error region
    errorbar="sd"
)
plt.title("ARI across Missing Fractions (All Methods)")
plt.ylabel("Adjusted Rand Index")
plt.xlabel("Missing Fraction")
plt.legend(title="Imputation Method", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
